In [1]:
import os
import sys

for _ in range(2):
    os.chdir("..")

os.getcwd()

'/Users/dananufrieva/MCS/six-00-twenty-search'

In [2]:
from dev.query_parser import *
from dev.posting_list import BasePostingList, PostingList, AntiPostingList
from dev.search_engine import SearchEngine

# Parser (Abstract syntax tree)

In [3]:
parser = QueryParser()

In [4]:
query = """
((sddas AND sdfds) OR sdfds) NOT asdfsa AND dfsdf NOT sdfsdfds AND dfdfd
"""

ast = parser.parse(query)
print(f"'{query}' -> {ast}")

'
((sddas AND sdfds) OR sdfds) NOT asdfsa AND dfsdf NOT sdfsdfds AND dfdfd
' -> AND(OR(AND(Term(sddas), Term(sdfds)), Term(sdfds)), Term(dfsdf), Term(dfdfd), NOT(Term(asdfsa)), NOT(Term(sdfsdfds)))


In [5]:
query = 'A OR (NOT B OR C OR NOT Z OR W)'

ast = parser.parse(query)
print(f"'{query}' -> {ast}")

'A OR (NOT B OR C OR NOT Z OR W)' -> OR(Term(A), OR(NOT(Term(B)), Term(C), NOT(Term(Z)), Term(W)))


# Execute 

In [6]:
q = 'apple OR banana'
ast = parser.parse(q)
print(f"'{q}' -> {ast}")

'apple OR banana' -> OR(Term(apple), Term(banana))


In [7]:
q = 'NOT (NOT (cat OR sobaka))'
ast = parser.parse(q)
print(f"'{q}' -> {ast}")

'NOT (NOT (cat OR sobaka))' -> NOT(NOT(OR(Term(cat), Term(sobaka))))


In [8]:
type(ast)

dev.query_parser.NotNode

In [9]:
type(ast.operand.operand)

dev.query_parser.OrNode

In [10]:
fuzzy_indexer = {
    'apple':  [1, 3, 5, 7, 9, 12, 13, 15, 16, 18],
    'banana': [2, 3, 4, 7, 8, 10, 13, 17, 19],
    'carrot': [1, 2, 4, 6, 8, 11, 14, 16, 19, 20],
    'date':   [3, 5, 6, 9, 10, 12, 15, 17, 18, 20],
    'even': [2, 4, 6, 8, 10, 12, 14, 16, 18, 20]
}

In [11]:
search_engine = SearchEngine(indexer=fuzzy_indexer)
search_engine.execute?

Signature: search_engine.execute(node: dev.query_parser.ASTNode) -> Union[dev.query_parser.ASTNode, dev.posting_list.BasePostingList]
Docstring: Execute AST
File:      ~/MCS/six-00-twenty-search/dev/search_engine.py
Type:      method

In [12]:
query_execute_tests = {
    # Simple OR
    'apple OR banana': f"({sorted(list(set(fuzzy_indexer['apple']) | set(fuzzy_indexer['banana'])))})",
    # Simple AND
    'apple AND banana': f"({sorted(list(set(fuzzy_indexer['apple']) & set(fuzzy_indexer['banana'])))})",
    # Triple OR
    'apple OR banana OR carrot': f"({sorted(list(set(fuzzy_indexer['apple']) | set(fuzzy_indexer['banana']) | set(fuzzy_indexer['carrot'])))})",
    # Triple AND
    'apple AND banana AND carrot': f"({sorted(list(set(fuzzy_indexer['apple']) & set(fuzzy_indexer['banana']) & set(fuzzy_indexer['carrot'])))})",
    # Mixed AND/OR with parentheses
    'apple AND (banana OR carrot)': f"({sorted(list(set(fuzzy_indexer['apple']) & (set(fuzzy_indexer['banana']) | set(fuzzy_indexer['carrot']))))})",
    # Mixed with commutativity
    '(apple OR banana) AND carrot': f"({sorted(list((set(fuzzy_indexer['apple']) | set(fuzzy_indexer['banana'])) & set(fuzzy_indexer['carrot'])))})",
    # AND with an empty intersection
    'banana AND date': f"({sorted(list(set(fuzzy_indexer['banana']) & set(fuzzy_indexer['date'])))})",
    # Single term
    'carrot': f"({sorted(list(set(fuzzy_indexer['carrot'])))})",
    # OR all
    'apple OR banana OR carrot OR date': f"({sorted(list(set(fuzzy_indexer['apple']) | set(fuzzy_indexer['banana']) | set(fuzzy_indexer['carrot']) | set(fuzzy_indexer['date'])))})",
    # AND none (intentional miss, expect empty)
    'carrot AND date AND apple AND banana': f"({sorted(list(set(fuzzy_indexer['apple']) & set(fuzzy_indexer['banana']) & set(fuzzy_indexer['carrot']) & set(fuzzy_indexer['date'])))})",
    # Simple NOT
    # 'NOT even': f"({sorted(list(set(range(1, 21)) - set(fuzzy_indexer['even'])))})",
    # AND (X NOT Y)
    'apple AND NOT banana': f"({sorted(list(set(fuzzy_indexer['apple']) - set(fuzzy_indexer['banana'])))})",
    # NOT (NOT X)
    'NOT (NOT apple)': f"({sorted(list(set(fuzzy_indexer['apple'])))})",
    # NOT(NOT(NOT(NOT X))
    'NOT(NOT(NOT(NOT apple)))': f"({sorted(list(set(fuzzy_indexer['apple'])))})",
    # X and NOT(NOT(NOT Y)
    'apple AND NOT(NOT(NOT banana))': f"({sorted(list(set(fuzzy_indexer['apple']) - set(fuzzy_indexer['banana'])))})",
    # OR :)
    'apple OR (apple OR (apple OR (apple OR (apple))))': f"({sorted(list(set(fuzzy_indexer['apple'])))})",
    'apple': f"({sorted(list(set(fuzzy_indexer['apple'])))})",
}


for q, res in query_execute_tests.items():
    ast = parser.parse(q)
    # print(f"'{q}' -> {ast}")
    executed = search_engine.execute(ast)
    assert str(executed) == query_execute_tests[q], f'Fail `{q} : {ast}`\n {str(executed)} != {query_execute_tests[q]}'
    
print('Success!')

Success!


In [13]:
query_execute_tests

{'apple OR banana': '([1, 2, 3, 4, 5, 7, 8, 9, 10, 12, 13, 15, 16, 17, 18, 19])',
 'apple AND banana': '([3, 7, 13])',
 'apple OR banana OR carrot': '([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20])',
 'apple AND banana AND carrot': '([])',
 'apple AND (banana OR carrot)': '([1, 3, 7, 13, 16])',
 '(apple OR banana) AND carrot': '([1, 2, 4, 8, 16, 19])',
 'banana AND date': '([3, 10, 17])',
 'carrot': '([1, 2, 4, 6, 8, 11, 14, 16, 19, 20])',
 'apple OR banana OR carrot OR date': '([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20])',
 'carrot AND date AND apple AND banana': '([])',
 'apple AND NOT banana': '([1, 5, 9, 12, 15, 16, 18])',
 'NOT (NOT apple)': '([1, 3, 5, 7, 9, 12, 13, 15, 16, 18])',
 'NOT(NOT(NOT(NOT apple)))': '([1, 3, 5, 7, 9, 12, 13, 15, 16, 18])',
 'apple AND NOT(NOT(NOT banana))': '([1, 5, 9, 12, 15, 16, 18])',
 'apple OR (apple OR (apple OR (apple OR (apple))))': '([1, 3, 5, 7, 9, 12, 13, 15, 16, 18])',
 'apple': '([1, 3,

# NEAR(terms*, k)

In [14]:
# Test for near_in
# Setup coordinate index for doc 1 with terms "a" and "b"
coord_index = {
    1: {
        'a': [1, 10, 15, 30],
        'b': [2, 12, 31]
    },
    2: {
        'a': [5, 20, 40],
        'b': [7, 25, 44]
    },
    3: {
        'a': [1, 2, 3],
        'b': [100]
    }
}

# Should return True: 'a' at 10, 'b' at 12 (distance 3, window k=3)
assert SearchEngine.near_in(coord_index, 1, ['a', 'b'], k=3) is True
assert SearchEngine.near_in(coord_index, 1, ['a', 'b'], k=2) is True
assert SearchEngine.near_in(coord_index, 1, ['a', 'b'], k=None) is True
# Should return False in doc 1 for k=1 (closest a=10, b=12)
assert SearchEngine.near_in(coord_index, 1, ['a', 'b'], k=1) is False
# Should return True for doc 2 with k=3 (a=20, b=25)
assert SearchEngine.near_in(coord_index, 2, ['a', 'b'], k=2) is False
assert SearchEngine.near_in(coord_index, 2, ['a', 'b'], k=6) is True
# Should return False in doc 3 (all a < b)
assert SearchEngine.near_in(coord_index, 3, ['a', 'b'], k=5) is False


print('Success!')



Success!
